# Read Streams

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_date, to_timestamp, expr
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

### **News**

In [0]:

df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "pkc-619z3.us-east1.gcp.confluent.cloud:9092") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option(
        "kafka.sasl.jaas.config",
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        "username='H42AXVVWCVXADIY4' "
        "password='cfltyQtMG9goqwVI8oPnbJ0sdX37LmByy4waREnCnpfC4+qyQm2MiqCYZK61n6VA';"
    ) \
    .option("subscribe", "news_topic") \
    .option("startingOffsets", "earliest") \
    .load()

df_final = df.selectExpr("CAST(value AS STRING)")

schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("title", StringType(), True),
    StructField("source", StringType(), True),
    StructField("publishedAt", StringType(), True),
    StructField("fetchedAt", StringType(), True)
])

from pyspark.sql.functions import from_json, col, to_timestamp

# --- 1. Parsing ---
news_parsed = df_final.withColumn("jsonData", from_json(col("value"), schema)) \
    .select("jsonData.*")

# --- 2. TRANSFORMATION ---
news = news_parsed \
    .withColumn("news_ts", to_timestamp(col("publishedAt"))) \
    .withWatermark("news_ts", "10 minutes")  # Kat-goulliha sber 10 min 3la l-khbar li m3etla


### **Trades**

In [0]:
df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "pkc-619z3.us-east1.gcp.confluent.cloud:9092") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option(
        "kafka.sasl.jaas.config",
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        "username='H42AXVVWCVXADIY4' "
        "password='cfltyQtMG9goqwVI8oPnbJ0sdX37LmByy4waREnCnpfC4+qyQm2MiqCYZK61n6VA';"
    ) \
    .option("subscribe", "trades_topic") \
    .option("startingOffsets", "earliest") \
    .load()

df_final = df.selectExpr("CAST(value AS STRING)")

schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", DoubleType(), True),
    StructField("event_time", DoubleType(), True)
])

trades = df_final.withColumn("jsonData", from_json(col("value"), schema)) \
    .select("jsonData.*") \
    .withColumn("trade_ts", to_timestamp(col("event_time").cast("timestamp"))) \
    .withWatermark("trade_ts", "10 minutes")

### **Fred**

In [0]:
df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "pkc-619z3.us-east1.gcp.confluent.cloud:9092") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option(
        "kafka.sasl.jaas.config",
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        "username='H42AXVVWCVXADIY4' "
        "password='cfltyQtMG9goqwVI8oPnbJ0sdX37LmByy4waREnCnpfC4+qyQm2MiqCYZK61n6VA';"
    ) \
    .option("subscribe", "fred_topic") \
    .option("startingOffsets", "earliest") \
    .load()

df_final = df.selectExpr("CAST(value AS STRING)")

schema = StructType([
    StructField("indicator", StringType(), True),
    StructField("series_id", StringType(), True),
    StructField("date", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("fetched_at", StringType(), True)
])

# 1. Parsing (hadchi dertih nâdy)
fred_parsed = df_final.withColumn("jsonData", from_json(col("value"), schema)) \
    .select("jsonData.*")

# 2. Transformation dyal l-Date (Darori bach t-jointiha m3a Trade)
fred = fred_parsed \
    .withColumn("fred_date", to_date(col("date"))) # m-String l-Date

# 3. Verif (haniya hit FRED Batch)
fred.show(20, truncate=False)

# Watermark

In [0]:
# 1. Transformi l-we9t (Timestamp) o t-dir l-Watermark
# ----------------------------------------------------
news_stream = news \
    .withColumn("news_ts", to_timestamp(col("publishedAt"))) \
    .withWatermark("news_ts", "10 minutes")

trades_stream = trades \
    .withColumn("trade_ts", to_timestamp(col("event_time").cast("timestamp"))) \
    .withWatermark("trade_ts", "10 minutes")

In [0]:
# 1. Join News + Trades b smiyat (Aliases) mferrqa
# -----------------------------------------------
enriched_market = trades_stream.alias("t").join(
    news_stream.alias("n"),
    expr("""
        t.symbol = n.symbol AND
        news_ts >= trade_ts - interval 5 minutes AND
        news_ts <= trade_ts + interval 5 minutes
    """),
    "inner"
).select("t.*", "n.title", "n.news_ts") # Hna kankhtaro ghir symbol wahed bach may-t-leleflch Spark mn be3d

In [0]:
# 3. Join m3a FRED (Batch-to-Stream)
# -----------------------------------
# Fred ghaliban batch, khass n-diro join 3la date
final_enriched_data = enriched_market.join(
    fred, 
    enriched_market.symbol == fred.indicator, 
    "left"
)

In [0]:
# 1. Config Kafka (Security)
kafka_security_options = {
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
                              "username='H42AXVVWCVXADIY4' "
                              "password='cfltyQtMG9goqwVI8oPnbJ0sdX37LmByy4waREnCnpfC4+qyQm2MiqCYZK61n6VA';"
}

# 2. Config Supabase (JDBC)
# Beddel 'your_project_id' b l-ID dyal l-projet dyalk o 'YOUR_PASSWORD'
supabase_url = "jdbc:postgresql://db.xxxxxxxx.supabase.co:5432/postgres" 
supabase_user = "postgres.xxxxxxxx"
supabase_password = "YOUR_PASSWORD"

In [0]:
def save_to_all_destinations(batch_df, batch_id):
    # Ila kan l-batch khawi, makhdamch
    if batch_df.isEmpty():
        return

    # a. Data Lake (Gold Layer)
    batch_df.write.format("parquet").mode("append").save("/mnt/invistis/gold/enriched_crypto_data")

    # b. Supabase (JDBC)
    try:
        batch_df.write.format("jdbc") \
            .option("url", supabase_url) \
            .option("dbtable", "enriched_market_data") \
            .option("user", supabase_user) \
            .option("password", supabase_password) \
            .option("driver", "org.postgresql.Driver") \
            .mode("append").save()
    except Exception as e:
        print(f"Error writing to Supabase: {e}")

    # c. Kafka Alerts (Ghir ila t-zad l-prix b 5% masalan)
    # Hna ghadi n-siftu koulchi dba bach n-testiw
    batch_df.selectExpr("CAST(symbol AS STRING) AS key", "to_json(struct(*)) AS value") \
        .write.format("kafka") \
        .option("kafka.bootstrap.servers", "pkc-619z3.us-east1.gcp.confluent.cloud:9092") \
        .option("topic", "alerts_topic") \
        .options(**kafka_security_options) \
        .save()